# mart_dev.daily_runs

A Saved table made by the batch daily_runs, with project.convert(...), from the statements in `conversion/statement.hql`, as its script wrote them. Its logic cell builds the same query from the Toolbox's own functions, with `create()`, `preview()` and `write()`.

## Terms

- **Saved table**: a table in the warehouse that only your own code writes, its files in a folder of its own.
- **Logic cell**: the code cell whose first line is `# logic`: the table's imports and functions alone.
- **Decision**: one thing about this table that only you decide, such as its location or its Key; `conversion/checklist.md` lists each one still open.
- **Key**: the columns whose values name exactly one row of the table.
- **Original**: the table the converted script writes, in the warehouse already, which this table copies, or, for a table made from a query, that query. `conversion/in_dev.hql` holds the original's Hive, which the cells comparing it read.

## Steps

1. **Answer its Decisions.** Goal: each Decision made. Action: answer each open Decision `conversion/checklist.md` lists in the decisions cell of `admin/workflows/convert/convert.ipynb`. Done when `conversion/checklist.md` says None under Open Decisions, once that Workflow's next step writes your answers in; then step 2.
2. **Run the cells.** Goal: its rows, its Hive and its lineage, tried. Action: run the code cells top to bottom. Done when each runs with no error.

## Decision rules

- If you still want to compare it with the original, keep the cells comparing them.
- If you don't, delete those cells: they stay until you delete them.

## Self-check

- `conversion/checklist.md` says None under Open Decisions.
- Each code cell runs top to bottom with no error.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # the Project: two up

from sqlglot_composer import (compare_hive, compare_with_original, export_hive, export_lineage,
                              project, run_query, show_hive)
from warehouse import query

In [ ]:
# logic
from sqlglot_composer import (
    AS,
    Clause,
    FROM,
    GROUP_BY,
    LEFT_JOIN,
    SELECT,
    Statement,
    WHERE,
    all_of,
    any_of,
    count_rows,
    derived,
    equals,
    fill_null,
    is_not_null,
    is_null,
    not_equals,
    statement,
    sum_of,
    unchecked_derived,
)
from references import ops
from sqlglot_composer import create_table  # for create()
from sqlglot_composer import INSERT_OVERWRITE, Statement, statement  # for write()
from references.mart_dev__daily_runs import daily_runs  # the Saved table's own Table reference

d = unchecked_derived("d", "SELECT\n  '2026-09-23' AS dt\nUNION ALL\nSELECT\n  '2026-09-24'", columns={
    "dt": "string",
}, key=["dt"])
job_runs = ops.job_runs
c = derived("c", statement(
    SELECT(
        job_runs.dt,
        AS(count_rows(), "runs"),
        AS(sum_of(job_runs.duration_mins), "total_duration_mins"),
    ),
    # CONVERSION: reads_all_partitions=True, as the original did; check it
    FROM(job_runs, reads_all_partitions=True),
    WHERE(all_of(
        is_not_null(job_runs.status),
        any_of(
            not_equals(job_runs.status, "TEST"),
            is_null(job_runs.status),
        ),
    )),
    GROUP_BY(job_runs.dt),
))


def rows() -> list[Clause]:
    """The clauses of the query, which converted() and a write share."""
    return [
        SELECT(
            d.dt,
            AS(fill_null(c.runs, 0), "runs"),
            AS(fill_null(c.total_duration_mins, 0), "total_duration_mins"),
        ),
        FROM(d),
        LEFT_JOIN(c, ON=equals(d.dt, c.dt)),
    ]


def converted() -> Statement:
    """The query, as the Hive this Conversion was made from writes it."""
    return statement(*rows())


def create() -> Statement:
    """Create mart_dev.daily_runs from its Table reference, if it isn't there yet."""
    return create_table(daily_runs, may_exist=True)


def preview() -> Statement:
    """The rows write() would save, as a SELECT to run and check first."""
    return converted()


def write() -> Statement:
    """Save the rows into mart_dev.daily_runs, as the original's INSERT did."""
    return statement(INSERT_OVERWRITE(daily_runs), *rows())


In [ ]:
# Try it: its rows, then, once its Table reference has its location=, the Hive of
# each step
run_query(preview(), query)
show_hive([create(), write()])

In [ ]:
# Lineage: a new pair in lineage/, named by the time and this table
export_lineage(write())

In [ ]:
# Its Hive: mart_dev__daily_runs.hql beside this notebook, from the logic as last saved
export_hive()

In [ ]:
# Its Hive against the original's, as conversion/in_dev.hql holds it
ORIGINAL = pathlib.Path("conversion/in_dev.hql").read_text(encoding="utf-8")
show_hive(preview())
print(compare_hive(preview(), ORIGINAL))

In [ ]:
# The convert Workflow's compare step, project.compare_batch, compares this table once
# written. This compares its query with the original's first:
# Its rows against the original's, through your query, on one Partition:
# print(compare_with_original(preview(), ORIGINAL, query=query, partition={"dt": "<a value>"}))

In [ ]:
# Its rows in the Reference sheet
project.refresh_reference_sheet("mart_dev.daily_runs")